In [1]:
import math
import numpy as np



In [2]:
# Hidden states
states = {"s": 0, "E": 1, "5": 2, "I": 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

# Transition probabilities
state_transition_prob = np.array([
    [0.0, 1.0, 0.0, 0.0, 0.0],
    [0.0, 0.9, 0.1, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 0.9, 0.1],
    [0.0, 0.0, 0.0, 0.0, 0.0]
])
# Emission probabilities
emission_nuc_codes = {
    'A': 0,
    'C': 1,
    'G': 2,
    'T': 3
}
emission_probs = np.array([
    [0.00, 0.00, 0.00, 0.00],
    [0.25, 0.25, 0.25, 0.25],
    [0.05, 0.00, 0.95, 0.00],
    [0.40, 0.10, 0.10, 0.40],
    [0.00, 0.00, 0.00, 0.00]
])
query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"



In [3]:
# log probability
def get_log_prob_for_state_path(state_path, query_sequence):
    prob = math.log(0.25)
    for i in range(1, len(state_path)):

        prev_state = states[state_path[i - 1]]
        curr_state = states[state_path[i]]

        transition = state_transition_prob[prev_state][curr_state]

        emission = emission_probs[
            curr_state
        ][emission_nuc_codes[query_sequence[i]]]
        prob += math.log(transition * emission)
    return prob



In [4]:
# Example manual paths
k1 = get_log_prob_for_state_path(
    "EEEEEE5IIIIIIIIIIIIIIIIIII",
    query_sequence
) + math.log(0.1)

print(k1)

k2 = get_log_prob_for_state_path(
    "EEEEEEEE5IIIIIIIIIIIIIIIII",
    query_sequence
) + math.log(0.1)

print(k2)



-43.89740030179307
-43.45111319916465


In [5]:
# Initialize Viterbi matrices
num_states = len(states)
seq_len = len(query_sequence)

viterbi_value_matrix = np.full(
    (num_states, seq_len),
    -np.inf
)

viterbi_trace_matrix = np.full(
    (num_states, seq_len),
    -1,
    dtype=int
)
# Initialize first column
first_nucleotide = query_sequence[0]

for state in range(num_states):
    transition = state_transition_prob[
        states["s"]
    ][state]

    emission = emission_probs[
        state
    ][emission_nuc_codes[first_nucleotide]]

    if transition > 0 and emission > 0:
        viterbi_value_matrix[state][0] = (
            math.log(transition)
            + math.log(emission)
        )
    viterbi_trace_matrix[state][0] = states["s"]



In [6]:
# Function to calculate one node
def calculate_prob_for_a_node(current_state, position):
    nucleotide = query_sequence[position]
    emission = emission_probs[
        current_state
    ][emission_nuc_codes[nucleotide]]
    if emission == 0:
        return -np.inf, -1

    best_prob = -np.inf
    best_prev_state = -1

    for prev_state in range(num_states):
        transition = state_transition_prob[
            prev_state
        ][current_state]

        if transition == 0:
            continue
        prev_prob = viterbi_value_matrix[
            prev_state
        ][position - 1]

        if prev_prob == -np.inf:
            continue

        current_prob = (
            prev_prob
            + math.log(transition)
            + math.log(emission)
        )
        if current_prob > best_prob:
            best_prob = current_prob
            best_prev_state = prev_state

    return best_prob, best_prev_state

# Fill Viterbi matrices
for col in range(1, seq_len):
    for row in range(num_states):
        best_prob, best_prev_state = (
            calculate_prob_for_a_node(
                row,
                col
            )
        )
        viterbi_value_matrix[row][col] = best_prob
        viterbi_trace_matrix[row][col] = best_prev_state

# Function to trace best path
def trace_state_path():

    last_column = viterbi_value_matrix[:, -1]
    best_last_state = np.argmax(last_column)
    best_probability = last_column[best_last_state]
    path = [best_last_state]
    current_state = best_last_state

    for col in range(seq_len - 1, 0, -1):
        previous_state = viterbi_trace_matrix[
            current_state
        ][col]
        path.append(previous_state)
        current_state = previous_state

    path.reverse()

    decoded_path = "".join(
        id2state[state]
        for state in path
    )
    return decoded_path, best_probability

best_path, best_prob = trace_state_path()



In [7]:
print("\nVITERBI VALUE MATRIX\n")
print(viterbi_value_matrix)
print("\nVITERBI TRACE MATRIX\n")
print(viterbi_trace_matrix)
print("\nQUERY SEQUENCE:")
print(query_sequence)
print("\nBEST STATE PATH:")
print(best_path)
print("\nBEST LOG PROBABILITY:")
print(best_prob)


VITERBI VALUE MATRIX

[[        -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf]
 [ -1.38629436  -2.87794924  -4.36960411  -5.86125899  -7.35291387
   -8.84456875 -10.33622362 -11.8278785  -13.31953338 -14.81118825
  -16.30284313 -17.79449801 -19.28615288 -20.77780776 -22.26946264
  -23.76111751 -25.25277239 -26.74442727 -28.23608214 -29.72773702
  -31.2193919  -32.71104677 -34.20270165 -35.69435653 -37.1860114
  -38.67766628]
 [        -inf         -inf         -inf         -inf -11.15957636
          -inf -11.19844713         -inf -14.18175689 -18.61785074
  -20.10950562 -21.6011605  -20.14837639         -inf -26.07612513
  -24.62334102 -29.05943488         -inf -29.09830565         -inf
  -35.02